# Analiza Przestrzenna Kielc - Zoptymalizowany Potok Danych

Celem tego notesu jest wczytanie surowych danych, filtracja do obszaru Kielc (za pomoca Bounding Box z Przystankow), transformacja układów współrzędnych do **PUWG 1992 (EPSG:2180)** oraz agregacja cen mieszkań do siatki populacyjnej 250m.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import os
from pathlib import Path

# --- KONFIGURACJA ŚCIEŻEK --- 
RAW_GUS = "zip://../data/raw/NSP2021_TOT_GRID250m_GPKG.zip!NSP2021_TOT_GRID250m.gpkg"
RAW_RCN = "../data/raw/rcn.gml"
INDEXED_RCN = "../data/processed/rcn_indexed.gpkg"
STOPS_PATH = "../data/raw/bus/stops.txt"
TARGET_CRS = "EPSG:2180"

print("Konfiguracja załadowana.")

Konfiguracja załadowana.


## 0. Optymalizacja GML -> GPKG

Konwersja wolnego formatu XML (GML) do zindeksowanej bazy danych (GeoPackage) za pomocą `ogr2ogr`.

In [2]:
if not os.path.exists(INDEXED_RCN):
    print("Optymalizacja GML -> GPKG (ogr2ogr)... to może chwilę potrwać.")
    # Konwertujemy warstwe Lokal (ceny) i Budynek (geometria budynkow)
    exit_code = os.system(f'ogr2ogr -f "GPKG" {INDEXED_RCN} {RAW_RCN} RCN_Lokal -nln prices')
    if exit_code != 0:
        print("Błąd podczas konwersji ogr2ogr!")
else:
    print("Plik zindeksowany RCN już istnieje.")

Plik zindeksowany RCN już istnieje.


## 1. Definicja Obszaru Analizy (Kielce Anchor)

Wyznaczamy maskę na podstawie przystanków autobusowych.

In [3]:
stops = pd.read_csv(STOPS_PATH)
stops_gdf = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs="EPSG:4326"
).to_crs(TARGET_CRS)

kielce_mask_2180 = box(*stops_gdf.total_bounds).buffer(3000)
print("Maska obszaru wyznaczona (bufor 3km).")

Maska obszaru wyznaczona (bufor 3km).


## 2. Chirurgiczny Odczyt: GUS NSP 2021

Wczytujemy tylko fragment siatki GUS (EPSG:3035).

In [4]:
bbox_3035 = gpd.GeoSeries([kielce_mask_2180], crs=TARGET_CRS).to_crs("EPSG:3035").total_bounds
nsp_kielce = gpd.read_file(RAW_GUS, bbox=tuple(bbox_3035), engine="pyogrio").to_crs(TARGET_CRS)
print(f"Wczytano {len(nsp_kielce)} komórek siatki GUS dla Kielc.")

Wczytano 34560 komórek siatki GUS dla Kielc.


## 3. Odczyt i Przygotowanie Cen: RCN

Wczytujemy z GPKG, filtrujemy ceny, liczymy wskaźniki przed transformacją.

In [5]:
bbox_2178 = gpd.GeoSeries([kielce_mask_2180], crs=TARGET_CRS).to_crs("EPSG:2178").total_bounds
prices_raw = gpd.read_file(INDEXED_RCN, bbox=tuple(bbox_2178), engine="pyogrio")

# Filtrowanie: Zostawiamy rekordy z cena (nawet bez metrazu - dla statystyk wolumenu)
prices_kielce = prices_raw.query('cenaLokaluBrutto > 0').copy()

# Bezpieczne obliczenie ceny za m2 (imputujemy NaN gdzie brak metrazu)
prices_kielce['price_per_m2'] = np.where(
    prices_kielce['powUzytkowaLokalu'] > 0, 
    prices_kielce['cenaLokaluBrutto'] / prices_kielce['powUzytkowaLokalu'],
    np.nan
)

prices_kielce = prices_kielce.to_crs(TARGET_CRS)
print(f"Wczytano {len(prices_kielce)} transakcji nieruchomości.")

Wczytano 8181 transakcji nieruchomości.


## 4. Agregacja do Siatki Analitycznej (Kwadraty 250m)

Łączymy ceny z siatką GUS, aby stworzyć profil ekonomiczny dzielnic.

In [6]:
print("Agregacja cen do siatki (Spatial Join)...")
joined = gpd.sjoin(prices_kielce, nsp_kielce, how="inner", predicate="within")

grid_stats = (
    joined.groupby("PL_code")
    .agg(
        mean_price_m2=("price_per_m2", "mean"),
        median_price_m2=("price_per_m2", "median"),
        mean_total_price=("cenaLokaluBrutto", "mean"),
        transaction_count=("cenaLokaluBrutto", "count")
    )
)

kielce_grid = nsp_kielce.merge(grid_stats, on="PL_code", how="left")
kielce_grid['transaction_count'] = kielce_grid['transaction_count'].fillna(0)

print(f"Siatka analityczna gotowa. Kwadratów z cenami: {len(grid_stats)} / {len(kielce_grid)}")

Agregacja cen do siatki (Spatial Join)...
Siatka analityczna gotowa. Kwadratów z cenami: 325 / 34560


## 5. Eksport Wyników

In [7]:
output_dir = Path("../data/processed")
kielce_grid.to_file(output_dir / "kielce_analytical_grid.gpkg", driver="GPKG")
prices_kielce.to_file(output_dir / "kielce_market_prices.gpkg", driver="GPKG")
stops_gdf.to_file(output_dir / "kielce_stops.gpkg", driver="GPKG")

print("Wszystkie pliki zapisane w data/processed/")

Wszystkie pliki zapisane w data/processed/
